# UNO simulation benchmark


Each cell executes one file from `scripts/` in the same kernel. 

The script entry point is `python run_simulation.py` and runs
the same stages in the same order.  

Run all stages for the complete analysis and NCO replacement experiment. To change
only the outcome, edit `scripts/04_single_outcome.py`, then rerun stages 4--7.
Changing data or Base settings requires rerunning the corresponding earlier stages.


## 0. Imports, fixed CPU settings, and numerical helpers.

Source: `scripts/00_setup.py`

Import the numerical helpers and set the same CPU and deterministic settings as the source.

In [ ]:
%run -i scripts/00_setup.py

## 1. Generate X, U, treatment and NCOs, then fix the NCO split.

Source: `scripts/01_generate_data.py`

Edit data parameters in this script. Treatment and NCO generation still use the `C`, `S`, and `V` risk functions.

In [ ]:
%run -i scripts/01_generate_data.py

## 2. Fit Logistic PS and train the Base neural network.

Source: `scripts/02_train_base.py`

Base uses width `100`, depth `3`, up to `50` epochs, and batch size `32`.  

In [ ]:
%run -i scripts/02_train_base.py

## 3. Estimate NCO-CATE signals, prune and fine-tune Base, then evaluate EASE.

Source: `scripts/03_run_uno.py`

UNO copies the fitted Base architecture and weights, uses post-ReLU activation ratios, and selects among `[0.35, 0.4, 0.45, 0.5]` using development EASE.

In [ ]:
%run -i scripts/03_run_uno.py

## 4. Generate Y and evaluate single-outcome RR, confidence intervals, and EC.

Source: `scripts/04_single_outcome.py`

The current outcome uses `Y_USE_V1 = True` and `Y_GAMMA_U = 2`. Both single-outcome EC/CI blocks execute in their existing order. EC variance is added on the log-RR scale.

In [ ]:
%run -i scripts/04_single_outcome.py

## 5. Evaluate the 1000 paired outcome replicates and empirical calibration.

Source: `scripts/05_repeated_outcomes.py`

Generate `1000` new outcome realizations for the same fixed cohort, treatments, NCOs, fitted models, and calibration parameters. These are outcome-only repetitions.

In [ ]:
%run -i scripts/05_repeated_outcomes.py

## 6. Run the retained development-NCO replacement experiment.

Source: `scripts/06_nco_replacement.py`

The retained setting is `NCO25_REPLACEMENT_FRACTION = 1` with the source seeded-random selection rule. This stage evaluates that one fraction, not a new fraction sweep. It preserves all original model/data integrity checks. Re-running this stage after editing its fraction accumulates completed scenarios in this session.

In [ ]:
%run -i scripts/06_nco_replacement.py

## 7. Save the current tables, arrays, software versions, and model weights.

Source: `scripts/07_save_results.py`

Write the computed tables, arrays and model states to `results/` in the working directory. Re-running this stage overwrites matching output filenames.  

In [ ]:
%run -i scripts/07_save_results.py